# Determine query batch size

This notebook compares active-learning query batch sizes by label efficiency, marginal validation improvement, acquisition behavior, and training dynamics. It discovers all completed runs under `runs/` automatically, so additional batch sizes and seeds appear without notebook changes.

Validation edit score is the decision metric (higher is better). Curves show individual seeds as faint lines and the across-seed mean as a solid line; shaded bands are ±1 standard deviation when replicated seeds are available.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

# Support execution from either the notebook directory or the repository root.
experiment_dir = Path.cwd()
if not (experiment_dir / "determine_query_batch_size.ipynb").exists():
    experiment_dir = experiment_dir / "experiments" / "2_determine_query_batch_size"
runs_dir = experiment_dir / "runs"

history_records, loss_records, config_records = [], [], []
for history_path in sorted(runs_dir.glob("*/history.json")):
    run_dir = history_path.parent
    with (run_dir / "config.json").open() as file:
        config = json.load(file)
    with history_path.open() as file:
        history = json.load(file)

    query_size = float(config["query_batch_size"])
    initial_size = float(config["initial_labeled_pool_size"])
    seed = int(config["seed"])
    config_records.append({
        "run": run_dir.name, "seed": seed, "query_size": query_size,
        "initial_size": initial_size, "strategy": config.get("query_strategy", "unknown"),
        "budget_unit": config.get("active_learning_budget_unit", "unknown"),
        "target_query_frames": config.get("query_batch_frame_budget", np.nan),
    })
    for row in history:
        history_records.append({
            "run": run_dir.name, "seed": seed, "query_size": query_size,
            "initial_size": initial_size, "round": int(row["round"]),
            "labeled_pool_size": int(row["labeled_pool_size"]),
            "labeled_pool_frames": int(row["labeled_pool_frames"]),
            "budget_pct": float(row["labeled_budget_percent"]),
            "best_epoch": int(row["best_epoch"]), "val_edit": float(row["val_edit"]),
        })
        loss_path = run_dir / f"round_{int(row['round']):03d}" / "loss.json"
        if loss_path.exists():
            with loss_path.open() as file:
                for loss in json.load(file):
                    loss_records.append({
                        "run": run_dir.name, "seed": seed, "query_size": query_size,
                        "round": int(row["round"]), "budget_pct": float(row["labeled_budget_percent"]),
                        "epoch": int(loss["epoch"]), "train_loss": float(loss["train"]),
                        "val_edit": np.nan if loss.get("val_edit") is None else float(loss["val_edit"]),
                    })

history_df = pd.DataFrame(history_records)
loss_df = pd.DataFrame(loss_records)
config_df = pd.DataFrame(config_records)
if history_df.empty:
    raise FileNotFoundError(f"No completed histories found under {runs_dir.resolve()}")
history_df = history_df.sort_values(["query_size", "seed", "round"]).reset_index(drop=True)
loss_df = loss_df.sort_values(["query_size", "seed", "round", "epoch"]).reset_index(drop=True)
query_sizes = sorted(history_df["query_size"].unique())
colors = dict(zip(query_sizes, plt.cm.viridis(np.linspace(0.12, 0.88, len(query_sizes)))))
print(f"Loaded {history_df['run'].nunique()} completed run(s), {len(history_df)} rounds, and "
      f"{len(loss_df):,} epoch records from {runs_dir.resolve()}")

## Run inventory

Before comparing curves, check coverage. A batch-size conclusion is reliable only when candidate sizes have comparable seed replication and reach comparable labeling budgets.

In [ ]:
inventory = (history_df.groupby("query_size")
             .agg(runs=("run", "nunique"), seeds=("seed", "nunique"),
                  min_budget_pct=("budget_pct", "min"), max_budget_pct=("budget_pct", "max"),
                  total_rounds=("round", "count"), mean_best_epoch=("best_epoch", "mean"))
             .reset_index())
display(inventory.style.format({
    "query_size": "{:.1f}%", "min_budget_pct": "{:.1f}%",
    "max_budget_pct": "{:.1f}%", "mean_best_epoch": "{:.1f}",
}))

## Label efficiency and diminishing returns

The left panel is the primary comparison: at the same labeled-frame budget, the better query size has the higher validation edit score. The right panel normalizes each round-to-round improvement by the additional labeled percentage; values near or below zero indicate a plateau.

In [ ]:
curve_summary = (history_df.groupby(["query_size", "round"])
                 .agg(budget_mean=("budget_pct", "mean"), edit_mean=("val_edit", "mean"),
                      edit_std=("val_edit", "std"), n=("run", "nunique"))
                 .reset_index())
marginal_df = history_df.copy()
marginal_df["budget_added_pct"] = marginal_df.groupby("run")["budget_pct"].diff()
marginal_df["edit_gain"] = marginal_df.groupby("run")["val_edit"].diff()
marginal_df["gain_per_added_pct"] = marginal_df["edit_gain"] / marginal_df["budget_added_pct"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
for query_size, group in history_df.groupby("query_size", sort=True):
    color = colors[query_size]
    for _, run_data in group.groupby("run"):
        axes[0].plot(run_data["budget_pct"], run_data["val_edit"], color=color, alpha=0.24, linewidth=1.2)
    summary = curve_summary[curve_summary["query_size"] == query_size]
    axes[0].plot(summary["budget_mean"], summary["edit_mean"], marker="o", color=color,
                 linewidth=2.4, markersize=4.5, label=f"{query_size:g}% (n={group['run'].nunique()})")
    std = summary["edit_std"].fillna(0)
    axes[0].fill_between(summary["budget_mean"], summary["edit_mean"] - std,
                         summary["edit_mean"] + std, color=color, alpha=0.13, linewidth=0)

    gain = marginal_df[(marginal_df["query_size"] == query_size) & marginal_df["gain_per_added_pct"].notna()]
    for _, run_data in gain.groupby("run"):
        axes[1].plot(run_data["budget_pct"], run_data["gain_per_added_pct"], color=color, alpha=0.24, linewidth=1.1)
    gain_summary = (gain.groupby("round")
                    .agg(budget_mean=("budget_pct", "mean"), gain_mean=("gain_per_added_pct", "mean"),
                         gain_std=("gain_per_added_pct", "std")).reset_index())
    axes[1].plot(gain_summary["budget_mean"], gain_summary["gain_mean"], marker="o",
                 color=color, linewidth=2.2, markersize=4.5, label=f"{query_size:g}%")
    gain_std = gain_summary["gain_std"].fillna(0)
    axes[1].fill_between(gain_summary["budget_mean"], gain_summary["gain_mean"] - gain_std,
                         gain_summary["gain_mean"] + gain_std, color=color, alpha=0.13, linewidth=0)

axes[0].set(title="Validation performance vs. labeling budget", xlabel="Labeled training frames (%)",
            ylabel="Validation edit score (higher is better)")
axes[0].legend(title="Query batch size")
axes[1].axhline(0, color="#555555", linewidth=1, linestyle="--")
axes[1].set(title="Marginal return of each acquisition", xlabel="Labeled training frames (%)",
            ylabel="Edit-score gain per added 1% frames")
axes[1].legend(title="Query batch size")
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## Acquisition behavior

Because samples contain different numbers of frames, the realized acquisition can differ slightly from the configured percentage. The first panel checks that discrepancy. The second shows how many newly labeled clips each acquisition requires—an annotation/selection overhead measure hidden by frame budget alone.

In [ ]:
acquisition_df = history_df.copy()
acquisition_df["frames_added"] = acquisition_df.groupby("run")["labeled_pool_frames"].diff()
acquisition_df["samples_added"] = acquisition_df.groupby("run")["labeled_pool_size"].diff()
acquisition_df["budget_added_pct"] = acquisition_df.groupby("run")["budget_pct"].diff()
acquisition_df = acquisition_df[acquisition_df["round"] > 0].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
rng = np.random.default_rng(7)
for query_size, group in acquisition_df.groupby("query_size", sort=True):
    jitter = rng.normal(0, max(0.03 * query_size, 0.03), len(group))
    axes[0].scatter(np.full(len(group), query_size) + jitter, group["budget_added_pct"],
                    color=colors[query_size], alpha=0.55, s=28)
    axes[0].scatter(query_size, group["budget_added_pct"].mean(), marker="D", s=60,
                    edgecolor="black", linewidth=0.7, color=colors[query_size], zorder=3)
    axes[1].scatter(group["budget_pct"], group["samples_added"], color=colors[query_size],
                    alpha=0.48, s=27, label=f"{query_size:g}%")
    sample_summary = group.groupby("round").agg(budget=("budget_pct", "mean"), samples=("samples_added", "mean")).reset_index()
    axes[1].plot(sample_summary["budget"], sample_summary["samples"], color=colors[query_size], linewidth=2)

limits = [0, max(query_sizes + acquisition_df["budget_added_pct"].dropna().tolist()) * 1.08]
axes[0].plot(limits, limits, color="#555555", linestyle="--", linewidth=1, label="configured = realized")
axes[0].set(xlim=limits, ylim=limits, title="Configured vs. realized query size",
            xlabel="Configured query batch (%)", ylabel="Frames added per acquisition (%)")
axes[0].legend()
axes[1].set(title="Clips added at each acquisition", xlabel="Labeled training frames (%)",
            ylabel="Newly labeled clips")
axes[1].legend(title="Query batch size")
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## Batch-size ranking

Normalized AULC is the average validation edit score over the observed budget interval, so it rewards performance throughout acquisition rather than only at the last round. Compare AULC only when candidates cover similar minimum and maximum budgets. `rounds_to_best` measures how quickly the best observed score is reached.

In [ ]:
run_metrics = []
for run, group in history_df.groupby("run"):
    group = group.sort_values("budget_pct")
    x = group["budget_pct"].to_numpy()
    y = group["val_edit"].to_numpy()
    width = x[-1] - x[0]
    best_position = int(np.argmax(y))
    run_metrics.append({
        "run": run, "query_size": group["query_size"].iloc[0], "seed": group["seed"].iloc[0],
        "rounds": len(group), "min_budget_pct": x[0], "max_budget_pct": x[-1],
        "normalized_aulc": np.trapz(y, x) / width if width > 0 else y[0],
        "final_edit": y[-1], "best_edit": y[best_position],
        "budget_at_best_pct": x[best_position], "rounds_to_best": best_position,
    })
run_metrics_df = pd.DataFrame(run_metrics)
ranking = (run_metrics_df.groupby("query_size")
           .agg(n_runs=("run", "count"), aulc_mean=("normalized_aulc", "mean"),
                aulc_std=("normalized_aulc", "std"), final_edit_mean=("final_edit", "mean"),
                final_edit_std=("final_edit", "std"), best_edit_mean=("best_edit", "mean"),
                budget_at_best_pct=("budget_at_best_pct", "mean"), rounds_to_best=("rounds_to_best", "mean"),
                min_budget_pct=("min_budget_pct", "min"), max_budget_pct=("max_budget_pct", "max"))
           .reset_index().sort_values("aulc_mean", ascending=False))
display(ranking.style.format({
    "query_size": "{:.1f}%", "aulc_mean": "{:.2f}", "aulc_std": "{:.2f}",
    "final_edit_mean": "{:.2f}", "final_edit_std": "{:.2f}", "best_edit_mean": "{:.2f}",
    "budget_at_best_pct": "{:.1f}%", "rounds_to_best": "{:.1f}",
    "min_budget_pct": "{:.1f}%", "max_budget_pct": "{:.1f}%",
}).background_gradient(subset=["aulc_mean", "final_edit_mean"], cmap="YlGn"))

## Epoch-level training diagnostics

Each row is one active-learning round. The loss heatmap reveals changes in convergence as the labeled pool grows; the validation heatmap shows both the delayed validation schedule and the epoch selected within each round. Rows are ordered by query size, seed, then round.

In [ ]:
diagnostics = loss_df.copy()
diagnostics["row"] = diagnostics.apply(
    lambda row: f"q={row['query_size']:g}% | s={int(row['seed'])} | r={int(row['round']):02d} | b={row['budget_pct']:.0f}%", axis=1
)
row_order = diagnostics.drop_duplicates(["query_size", "seed", "round"])["row"].tolist()
train_heatmap = diagnostics.pivot(index="row", columns="epoch", values="train_loss").reindex(row_order)
edit_heatmap = diagnostics.pivot(index="row", columns="epoch", values="val_edit").reindex(row_order)

height = max(5.5, 0.32 * len(row_order) + 1.8)
fig, axes = plt.subplots(1, 2, figsize=(15, height), sharey=True)
image_loss = axes[0].imshow(train_heatmap, aspect="auto", interpolation="nearest", cmap="magma_r")
masked_edit = np.ma.masked_invalid(edit_heatmap.to_numpy(dtype=float))
edit_cmap = plt.cm.viridis.copy()
edit_cmap.set_bad("#eeeeee")
image_edit = axes[1].imshow(masked_edit, aspect="auto", interpolation="nearest", cmap=edit_cmap)

tick_step = max(1, len(train_heatmap.columns) // 10)
ticks = np.arange(0, len(train_heatmap.columns), tick_step)
for ax, title in zip(axes, ["Training loss by epoch", "Validation edit score by epoch"]):
    ax.set(title=title, xlabel="Epoch")
    ax.set_xticks(ticks, train_heatmap.columns[ticks])
axes[0].set_yticks(np.arange(len(row_order)), row_order, fontsize=8)
axes[0].set_ylabel("Query size | seed | round | labeled budget")
fig.colorbar(image_loss, ax=axes[0], fraction=0.025, pad=0.02, label="Training loss")
fig.colorbar(image_edit, ax=axes[1], fraction=0.025, pad=0.02, label="Validation edit")
fig.tight_layout()
plt.show()

## Current evidence

Use the generated inventory and ranking above when more runs are added. With fewer than two seeds for a batch size, variability and confidence cannot be estimated; with only one tested batch size, the notebook describes that learning trajectory but cannot establish that it is optimal. A defensible selection should prioritize replicated AULC at matched budget coverage, then use marginal returns and acquisition overhead as tie-breakers.